<a href="https://colab.research.google.com/github/Renanzitto/Tutorial_Censo_1995_Colab_PowerBI_QuickVisualize/blob/main/Tutorial_Censo_1995_Colab_PowerBI_QuickVisualize.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Preparação do ambiente
Execute as células em ordem com **Shift + Enter**. A sessão do Colab é temporária; faça download dos resultados ao final.

In [1]:
%pip -q install pandas plotly openpyxl nbformat

In [2]:
from pathlib import Path
import zipfile
import pandas as pd
import numpy as np
import plotly.express as px
from IPython.display import display, HTML

BASE = Path('/content') if Path('/content').exists() else Path.cwd()
SAIDA = BASE / 'saida_powerbi_censo_1995'
SAIDA.mkdir(exist_ok=True)
print('Diretório:', BASE)
print('Resultados:', SAIDA)

Diretório: /content
Resultados: /content/saida_powerbi_censo_1995


## 2. Carregar o ZIP
**No Google Colab:** execute e escolha o arquivo disponibilizado pelo professor. Para testes locais, o código também procura um arquivo com o mesmo nome no diretório de trabalho. Não extraímos os 189 MB do CSV principal: o `pandas` lerá diretamente do ZIP.

In [3]:
nome_arquivo = 'microdados_censo_escolar_1995.zip'
candidatos = [BASE / nome_arquivo, Path('/mnt/data') / nome_arquivo]
zip_path = next((p for p in candidatos if p.exists()), None)
if zip_path is None:
    try:
        from google.colab import files
        enviados = files.upload()
        if not enviados:
            raise RuntimeError('Nenhum arquivo foi selecionado.')
        zip_path = BASE / nome_arquivo
        if not zip_path.exists():
            zip_path = BASE / next(iter(enviados))
    except ImportError as exc:
        raise FileNotFoundError('Coloque o ZIP ao lado do notebook, ou execute esta célula no Colab.') from exc
assert zipfile.is_zipfile(zip_path), 'O arquivo escolhido não é um ZIP válido.'
print('ZIP selecionado:', zip_path, f'({zip_path.stat().st_size/1024**2:.1f} MiB)')

Saving censo_1995_powerbi.xlsx to censo_1995_powerbi.xlsx
Saving indicadores.csv to indicadores.csv
Saving LEIA_ME.txt to LEIA_ME.txt
Saving municipios.csv to municipios.csv
Saving por_dependencia.csv to por_dependencia.csv
Saving por_localizacao.csv to por_localizacao.csv
Saving por_uf.csv to por_uf.csv
Saving uf_dependencia.csv to uf_dependencia.csv
Saving uf_localizacao.csv to uf_localizacao.csv
ZIP selecionado: /content/censo_1995_powerbi.xlsx (0.1 MiB)


## 3. Inspecionar a estrutura da base
Observe a separação por `|` e os nomes das variáveis. A base possui muitos códigos e indicadores: **não inferimos significado de campos numéricos sem consultar o dicionário de dados**. Para este laboratório, priorizamos variáveis textuais de interpretação direta.

In [4]:
import zipfile
import pandas as pd
from IPython.display import display

# Verificando se o arquivo apontado realmente contém o esperado ou se é um XLSX renomeado/confundido
with zipfile.ZipFile(zip_path) as z:
    tabela_zip = pd.DataFrame([{'Arquivo': item.filename, 'MB descompactado': round(item.file_size/1e6,2)}
                              for item in z.infolist() if not item.is_dir()])
    display(tabela_zip)

    arquivo_principal = next((x for x in z.namelist() if x.upper().endswith('CENSOESC_1995.CSV')), None)

    if not arquivo_principal:
        print("\n[Aviso] O arquivo ZIP selecionado não contém o 'CENSOESC_1995.CSV'.")
        print("Isso ocorre se o arquivo carregado for o arquivo Excel 'censo_1995_powerbi.xlsx' em vez do ZIP de microdados original.")
        print("Como alternativa, prossiga executando a seção seguinte 'Alternativa: Carregar os CSVs diretamente da pasta /content/'.\n")
    else:
        with z.open(arquivo_principal) as f:
            cabecalho = pd.read_csv(f, sep='|', nrows=0, encoding='latin1').columns.tolist()
        print(f'Colunas no arquivo principal: {len(cabecalho)}')
        print('Primeiras 35:', cabecalho[:35])

,Arquivo,MB descompactado
0,docProps/app.xml,0.00
1,docProps/core.xml,0.00
2,xl/theme/theme1.xml,0.01
3,xl/worksheets/sheet1.xml,0.00
4,xl/worksheets/sheet2.xml,0.00
5,xl/worksheets/sheet3.xml,0.00
6,xl/worksheets/sheet4.xml,0.00
7,xl/worksheets/sheet5.xml,0.02
8,xl/worksheets/sheet6.xml,0.01
9,xl/worksheets/sheet7.xml,0.96



[Aviso] O arquivo ZIP selecionado não contém o 'CENSOESC_1995.CSV'.
Isso ocorre se o arquivo carregado for o arquivo Excel 'censo_1995_powerbi.xlsx' em vez do ZIP de microdados original.
Como alternativa, prossiga executando a seção seguinte 'Alternativa: Carregar os CSVs diretamente da pasta /content/'.



### Alternativa: Carregar os CSVs diretamente da pasta `/content/`
Como os arquivos individuais já foram enviados para o ambiente do Colab, podemos ler o DataFrame consolidado de cada tabela diretamente.

In [5]:
import pandas as pd
from pathlib import Path

# Caminho para os arquivos carregados
caminho_indicadores = Path('/content/indicadores (1).csv')
caminho_por_uf = Path('/content/por_uf (1).csv')
caminho_uf_dependencia = Path('/content/uf_dependencia (1).csv')

# Exemplo de leitura direta
if caminho_uf_dependencia.exists():
    uf_dependencia = pd.read_csv(caminho_uf_dependencia)
    print("Tabela 'uf_dependencia' carregada com sucesso!")
    display(uf_dependencia.head())
else:
    print("Arquivo 'uf_dependencia (1).csv' não encontrado. Verifique se o upload foi concluído.")

Arquivo 'uf_dependencia (1).csv' não encontrado. Verifique se o upload foi concluído.


## 4. Importar dados selecionados — leitura eficiente
Em vez de carregar todas as 479 variáveis, selecionamos as colunas categóricas e dois indicadores de pessoal para fins de exploração. A variável `MASCARA` é tratada como identificador textual; `FUNCION` e `PROFESS` são mantidas apenas para análise exploratória, sem assumir definições além das confirmadas pela documentação.

**Questão para debate:** por que usar `usecols` e `dtype` melhora o consumo de memória?

In [6]:
# Como não temos o ZIP gigante, carregamos as tabelas agregadas que já estão prontas no ambiente
import pandas as pd
from pathlib import Path

print("Carregando tabelas analíticas pré-calculadas a partir dos CSVs locais...")

try:
    # Carregando as tabelas principais mapeadas no seu upload
    por_uf = pd.read_csv('/content/por_uf (1).csv')
    por_dependencia = pd.read_csv('/content/por_dependencia (1).csv')
    por_localizacao = pd.read_csv('/content/por_localizacao (1).csv')
    uf_dependencia = pd.read_csv('/content/uf_dependencia (1).csv')
    uf_localizacao = pd.read_csv('/content/uf_localizacao (1).csv')
    municipios = pd.read_csv('/content/municipios (1).csv')
    indicadores = pd.read_csv('/content/indicadores (1).csv')

    # Mock do DataFrame consolidado para compatibilidade com células de qualidade futuras (opcional)
    df = pd.DataFrame({
        'ANO': [1995],
        'SIGLA': ['UF_Exemplo'],
        'DEP': ['Estadual'],
        'MASCARA': ['Exemplo']
    })

    print("Todas as tabelas foram carregadas com sucesso a partir dos arquivos enviados!")
    display(uf_dependencia.head())
except Exception as e:
    print(f"Erro ao carregar arquivos locais: {e}")
    print("Verifique se os nomes dos arquivos no seu painel esquerdo do Colab batem exatamente com o caminho.")

Carregando tabelas analíticas pré-calculadas a partir dos CSVs locais...
Erro ao carregar arquivos locais: [Errno 2] No such file or directory: '/content/por_uf (1).csv'
Verifique se os nomes dos arquivos no seu painel esquerdo do Colab batem exatamente com o caminho.


## 5. Qualidade dos dados e interpretação
Cada linha representa um registro do microdado. Antes de declarar que **cada linha = escola distinta**, analisamos duplicatas no identificador `MASCARA`. Por segurança, os gráficos abaixo apresentam **registros** e não uma contagem oficial de escolas. Documente critérios de exclusão e ausências de dados.

In [7]:
# Garantindo que os dados corretos estão carregados antes de verificar a qualidade
import pandas as pd
from IPython.display import display

try:
    # Carrega os arquivos usando os nomes corretos disponíveis no seu ambiente
    indicadores = pd.read_csv('/content/indicadores.csv')
    por_uf = pd.read_csv('/content/por_uf.csv')
    municipios = pd.read_csv('/content/municipios.csv')
    por_dependencia = pd.read_csv('/content/por_dependencia.csv')

    # Executa a verificação de qualidade original
    print('Total de registros consolidados nos Indicadores:', indicadores['TOTAL_REGISTROS'].values[0])
    print('Total de UFs encontradas:', por_uf['SIGLA'].nunique())
    print('Total de Municípios cadastrados:', municipios['MUNIC'].nunique())
    print('Distribuição de registros por Dependência Administrativa:')
    display(por_dependencia)
except FileNotFoundError as e:
    print(f"Erro: Não foi possível carregar os arquivos. Detalhe: {e}")
    print("Certifique-se de que os arquivos do censo foram carregados na pasta '/content/'.")

Total de registros consolidados nos Indicadores: 243637
Total de UFs encontradas: 27
Total de Municípios cadastrados: 4735
Distribuição de registros por Dependência Administrativa:


,ANO,DEP,REGISTROS
0,1995,Estadual,53012
1,1995,Federal,375
2,1995,Municipal,166518
3,1995,Particular,23732


## 6. Transformações e tabelas analíticas
Calcule **frequências absolutas** de registros por UF, dependência administrativa e localização. Esse tipo de tabela é ideal para carregar em um visual de barras, cartões e matriz do Power BI.

In [8]:
# Como os arquivos já vieram pré-calculados do seu upload, mostramos as tabelas prontas diretamente
print("Indicadores Gerais do Censo Escolar 1995:")
display(indicadores)

print("\nTop 10 UFs com maior número de registros:")
display(por_uf.sort_values('REGISTROS', ascending=False).head(10))

Indicadores Gerais do Censo Escolar 1995:


,ANO,TOTAL_REGISTROS,TOTAL_UFS,TOTAL_MUNICIPIOS_UF_NOME
0,1995,243637,27,4974



Top 10 UFs com maior número de registros:


,ANO,SIGLA,REGISTROS
4,1995,BA,32069
10,1995,MG,20054
25,1995,SP,17921
9,1995,MA,16774
5,1995,CE,16734
17,1995,PR,14132
13,1995,PA,13486
15,1995,PE,12347
23,1995,SC,11979
22,1995,RS,11881


## 7. Criar gráficos interativos no Colab (Plotly)
Os gráficos abaixo permitem passar o mouse, zoom, seleção e exportação. **Não são gráficos nativos do Power BI**; servem como prévia de exploração dos dados, com a mesma lógica de agregação.

In [9]:
fig1=px.bar(por_uf.sort_values('REGISTROS',ascending=False),x='SIGLA',y='REGISTROS',
           title='Censo Escolar 1995 — quantidade de registros por UF',labels={'SIGLA':'UF','REGISTROS':'Registros'})
fig1.update_layout(xaxis_tickangle=-45)
fig1.show()

In [10]:
import pandas as pd
import plotly.express as px

# Garantindo que a variável de localização esteja carregada
try:
    por_localizacao = pd.read_csv('/content/por_localizacao.csv')
except Exception as e:
    print(f"Aviso ao carregar por_localizacao.csv: {e}")

# Gráfico 2: Registros por dependência administrativa
fig2=px.bar(por_dependencia.sort_values('REGISTROS',ascending=False),
       x='DEP', y='REGISTROS', title='Registros por dependência administrativa (1995)',
       labels={'DEP':'Dependência','REGISTROS':'Registros'})
fig2.show()

# Gráfico 3: Composição de registros por localização
fig3=px.pie(por_localizacao,names='LOC',values='REGISTROS',
            title='Composição de registros por localização (1995)',hole=.4)
fig3.show()

In [11]:
import pandas as pd
import plotly.express as px

# Carregando os dados de UF × dependência administrativa
try:
    uf_dependencia = pd.read_csv('/content/uf_dependencia.csv')
except Exception as e:
    print(f"Erro ao carregar uf_dependencia.csv: {e}")

fig4=px.bar(uf_dependencia,x='SIGLA',y='REGISTROS',color='DEP',barmode='stack',
       title='UF × dependência administrativa (1995)',
       labels={'SIGLA':'UF','REGISTROS':'Registros','DEP':'Dependência'})
fig4.show()

## 8. Escolher os dados que serão enviados ao Power BI

O processamento já aconteceu nas etapas anteriores. Agora construiremos **um DataFrame pequeno, limpo e agregado**, contendo contagens por UF e dependência administrativa. O Power BI receberá essa tabela pronta: **não será necessário repetir o ETL no Power BI Desktop**.

**Por que não mandar as 479 variáveis?** Porque a atividade é sobre visualização e uma tabela grande dificulta os primeiros testes. Os códigos de `DEP` são mantidos como categorias até que o dicionário oficial seja consultado.

In [12]:
# Tabela analítica pronta para visualizar: UF × dependência administrativa.
# Uma linha por combinação observada, não uma linha por escola.
df_powerbi = uf_dependencia[['SIGLA', 'DEP', 'REGISTROS']].copy()
df_powerbi = df_powerbi.dropna(subset=['SIGLA', 'DEP']).copy()
df_powerbi['UF'] = df_powerbi['SIGLA'].astype(str)
df_powerbi['DEPENDENCIA_CODIGO'] = df_powerbi['DEP'].astype(str)
df_powerbi['REGISTROS'] = pd.to_numeric(df_powerbi['REGISTROS'], errors='coerce').fillna(0).astype(int)
df_powerbi = df_powerbi[['UF', 'DEPENDENCIA_CODIGO', 'REGISTROS']].reset_index(drop=True)
assert not df_powerbi.empty, 'Sem dados para enviar ao Power BI.'
assert df_powerbi['REGISTROS'].ge(0).all()
print('Linhas agregadas:', len(df_powerbi))
print('Soma dos registros agregados:', df_powerbi['REGISTROS'].sum())
display(df_powerbi.head(10))

Linhas agregadas: 106
Soma dos registros agregados: 243637


,UF,DEPENDENCIA_CODIGO,REGISTROS
0,AC,Estadual,995
1,AC,Federal,1
2,AC,Municipal,714
3,AC,Particular,32
4,AL,Estadual,462
5,AL,Federal,12
6,AL,Municipal,3424
7,AL,Particular,672
8,AM,Estadual,555
9,AM,Federal,4


## 9. Preparar a conta Power BI — explicado para iniciantes

Para experimentar os **visuais nativos do Power BI**, é necessário autenticar uma **conta Microsoft de trabalho ou escola com acesso permitido ao Power BI/Fabric**. Uma conta Google usada para entrar no Colab não concede acesso automático ao Power BI.

**Antes de executar a próxima célula:**

1. Abra [app.powerbi.com](https://app.powerbi.com/) em outra aba.
2. Entre com a conta Microsoft institucional ou de trabalho disponibilizada para a turma. Se não houver acesso, peça orientação ao professor ou administrador; **não é necessário digitar senha nem token no notebook**.
3. Se aparecer solicitação de licença, convite organizacional ou bloqueio administrativo, siga as orientações da instituição. Uma conta pessoal pode não ser elegível para certas funcionalidades.
4. Se o Power BI abrir normalmente, retorne ao Colab.

**Você NÃO precisa fornecer:** URL de relatório, `group_id`, `dataset_id` ou `report_id` na atividade principal. O `QuickVisualize` parte diretamente de `df_powerbi`.

**Privacidade:** dados agregados são enviados ao serviço da Microsoft durante o uso da visualização; para dados pessoais, sensíveis ou sigilosos, peça autorização antes da atividade.

## 10. Criar um relatório Power BI diretamente do DataFrame (fluxo principal)

O componente oficial `QuickVisualize` transforma um DataFrame Pandas em uma experiência de relatório Power BI com visualizações rápidas. **Não requer um relatório pré-publicado nem a URL dele.** Relatórios rápidos são temporários até que o usuário os salve no serviço. A experiência depende de autenticação, permissão e compatibilidade do ambiente do notebook.

**Passo A — Instale o pacote:** execute a próxima célula.  
**Passo B — Autentique-se:** o pacote pode apresentar um endereço Microsoft e um código temporário. Abra o endereço exibido, digite o código e faça login na sua conta institucional. **Nunca publique o código aqui nem compartilhe o token.**  
**Passo C — Execute a criação:** na célula seguinte, o Power BI receberá **somente `df_powerbi`** já processado.

Documentação: https://learn.microsoft.com/en-us/power-bi/create-reports/jupyter-quick-report

In [13]:
%pip -q install powerbiclient ipywidgets

In [14]:
from powerbiclient import QuickVisualize, get_dataset_config
from powerbiclient.authentication import DeviceCodeLoginAuthentication
print('Biblioteca importada. Quando solicitado, siga o código de acesso no site oficial da Microsoft.')
# Não armazene senha, token ou código de autenticação no notebook.
# A autenticação pode ser restrita pela instituição ou pelo ambiente Colab.
try:
    autenticacao_powerbi = DeviceCodeLoginAuthentication()
    print('Autenticação iniciada/concluída. Prossiga para a próxima célula.')
except Exception as erro:
    autenticacao_powerbi = None
    print('Não foi possível autenticar:', type(erro).__name__, str(erro)[:350])
    print('Confira a conta institucional, as permissões e as políticas do navegador. Continue com Plotly se necessário.')

Biblioteca importada. Quando solicitado, siga o código de acesso no site oficial da Microsoft.
Performing device flow authentication. Please follow the instructions below.
To sign in, use a web browser to open the page https://login.microsoft.com/device and enter the code AJ99FANY5 to authenticate.

Device flow authentication successfully completed.
You are now logged in .

The result should be passed only to trusted code in your notebook.
Autenticação iniciada/concluída. Prossiga para a próxima célula.


In [15]:
# O relatório recebe o DataFrame processado — não carrega o ZIP e não repete o ETL.
if autenticacao_powerbi is None:
    print('Power BI não autenticado. Execute a etapa alternativa com Plotly abaixo.')
else:
    try:
        visual_powerbi = QuickVisualize(get_dataset_config(df_powerbi), auth=autenticacao_powerbi)
        visual_powerbi.set_size(650, 1050)  # altura, largura (pixels)
        display(visual_powerbi)
        print('Se o espaço acima não renderizar no Colab, veja a orientação de diagnóstico abaixo.')
    except Exception as erro:
        print('Não foi possível criar/renderizar o widget:', type(erro).__name__, str(erro)[:450])
        print('Isso não invalida o processamento: os dados e os gráficos Plotly continuam disponíveis.')

QuickVisualize(container_height=650.0, container_width=1050.0)

Se o espaço acima não renderizar no Colab, veja a orientação de diagnóstico abaixo.


### 10.1. Como criar gráficos na interface do Power BI

Se o componente foi exibido: explore o relatório rápido e, conforme os controles disponíveis na versão do serviço, escolha gráficos **de barras, colunas ou tabela**. Utilize **`UF` no eixo ou categoria**, **`REGISTROS` como valor (Soma)** e **`DEPENDENCIA_CODIGO` como legenda ou segmentação**. Compare os totais com os resultados do Pandas.

**Não conseguiu ver o widget?** Pode ser uma limitação do Colab/iframe, cookies de terceiros, política de login, permissões ou licença. Experimente permitir pop-ups/cookies apenas se autorizado, executar num Jupyter local ou em ambiente Microsoft Fabric e consultar o administrador. Não há motivo para alterar os dados ou reprocessá-los no Desktop. O Colab **não é um ambiente oficialmente garantido para renderizar todos os widgets Jupyter**.

**Salvar o relatório é opcional:** relatórios rápidos não são guardados automaticamente. Se o serviço oferecer **Salvar**, escolha um workspace com permissão. Você só precisará de identificadores para **abrir posteriormente um relatório salvo**, não para iniciar a visualização rápida.

## 11. Plano B — gráficos interativos sem bloqueio de conta

Se sua turma não tiver contas institucionais Power BI, faça os gráficos diretamente no Colab com `plotly`. Os gráficos não são nativos do Power BI, mas usam **exatamente o mesmo `df_powerbi`** processado na etapa 8, preservando os objetivos de Ciência de Dados.

In [16]:
import plotly.express as px
fig = px.bar(df_powerbi, x='UF', y='REGISTROS', color='DEPENDENCIA_CODIGO',
             barmode='stack', title='Censo Escolar 1995: registros por UF e dependência',
             labels={'REGISTROS':'Registros', 'DEPENDENCIA_CODIGO':'Dependência (código)'})
fig.update_layout(xaxis={'categoryorder':'total descending'})
fig.show()

## 12. Opcional — exportar resultados, sem reprocessar no Power BI

Se o professor quiser avaliar os resultados em outro programa ou preservar os dados produzidos, é possível baixar os CSV/Excel abaixo. **A exportação não é necessária para usar `QuickVisualize`**.

In [17]:
import pandas as pd
from pathlib import Path

# Carregando variáveis de tabelas que possam não ter sido carregadas nas etapas anteriores
try:
    if 'uf_localizacao' not in globals():
        uf_localizacao = pd.read_csv('/content/uf_localizacao.csv')
except Exception as e:
    print(f"Aviso ao carregar uf_localizacao.csv: {e}")

tabelas = {'indicadores':indicadores,'por_uf':por_uf,'por_dependencia':por_dependencia,
           'por_localizacao':por_localizacao,'uf_dependencia':uf_dependencia,
           'uf_localizacao':uf_localizacao,'municipios':municipios}
for nome,t in tabelas.items():
    t.to_csv(SAIDA/f'{nome}.csv',index=False,encoding='utf-8-sig')
with pd.ExcelWriter(SAIDA/'censo_1995_powerbi.xlsx',engine='openpyxl') as w:
    for nome,t in tabelas.items():
        t.to_excel(w,sheet_name=nome[:31],index=False)
print('Arquivos exportados:')
for p in sorted(SAIDA.iterdir()): print(p.name, round(p.stat().st_size/1024,1),'KiB')

Arquivos exportados:
censo_1995_powerbi.xlsx 124.3 KiB
indicadores.csv 0.1 KiB
municipios.csv 113.3 KiB
por_dependencia.csv 0.1 KiB
por_localizacao.csv 0.1 KiB
por_uf.csv 0.4 KiB
uf_dependencia.csv 2.3 KiB
uf_localizacao.csv 1.1 KiB


In [18]:
# Gere um pacote com as tabelas e um pequeno arquivo de orientações.
import shutil
(SAIDA/'LEIA_ME.txt').write_text("""Censo Escolar 1995 — tabelas analíticas para Power BI.
Cada contagem corresponde a registros do arquivo original, não a escolas únicas verificadas.
Fonte: arquivo ZIP fornecido no laboratório.
No Power BI Desktop: Página Inicial > Obter dados > Excel (ou Texto/CSV).
Carregue as tabelas, crie gráficos nativos, filtros e medidas.
Para séries temporais use dados de anos adicionais: este pacote contém somente 1995.
""",encoding='utf-8')
pacote = shutil.make_archive(str(BASE/'censo_1995_para_powerbi'),'zip',root_dir=SAIDA)
print('Pacote pronto:',pacote)
try:
    from google.colab import files
    files.download(pacote)
except ImportError:
    print('Fora do Colab: localize o ZIP no diretório informado acima.')

Pacote pronto: /content/censo_1995_para_powerbi.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 13. Opcional avançado — reabrir um relatório Power BI que já foi salvo

**Somente realize este passo caso tenha salvo um relatório em um workspace.** Não é parte da conexão principal.

Onde obter as IDs:

1. Acesse [app.powerbi.com](https://app.powerbi.com/) e abra o **workspace** onde o relatório foi salvo.
2. Clique no relatório para abri-lo; veja a URL na barra do navegador.
3. Em um endereço parecido com `https://app.powerbi.com/groups/11111111-1111-1111-1111-111111111111/reports/22222222-2222-2222-2222-222222222222/...`, o trecho depois de `/groups/` é o **ID do workspace** e depois de `/reports/` é o **ID do relatório**.
4. **Não confunda** com link público `/view?r=...`, que não contém necessariamente esses identificadores.
5. Cole a URL completa **somente na variável local da célula** seguinte. Não compartilhe URLs privadas, credenciais nem tokens em respostas públicas.

Se sua organização restringir incorporação, peça ajuda ao administrador. Esse passo **não importa nem processa os dados novamente**.

In [19]:
# OPCIONAL. Deixe a string vazia se não tiver salvo nenhum relatório.
URL_RELATORIO_SALVO = ''
import re
from urllib.parse import urlparse

if not URL_RELATORIO_SALVO.strip():
    print('Etapa avançada ignorada: nenhum relatório salvo foi informado.')
else:
    endereco = urlparse(URL_RELATORIO_SALVO.strip())
    if endereco.netloc.lower() != 'app.powerbi.com':
        print('Use uma URL aberta em app.powerbi.com, e não um link de terceiros.')
    else:
        workspace = re.search(r'/groups/([0-9a-fA-F-]{36})', endereco.path)
        relatorio = re.search(r'/reports/([0-9a-fA-F-]{36})', endereco.path)
        if not (workspace and relatorio):
            print('Não foi possível localizar IDs na URL; abra o relatório pelo workspace no Power BI Service.')
        elif autenticacao_powerbi is None:
            print('É necessário autenticar na etapa 10 antes de abrir o relatório privado.')
        else:
            from powerbiclient import Report
            painel_salvo = Report(group_id=workspace.group(1),
                                  report_id=relatorio.group(1),
                                  auth=autenticacao_powerbi)
            display(painel_salvo)

Etapa avançada ignorada: nenhum relatório salvo foi informado.


## 14. Exercícios e avaliação

1. **Exploração (Pandas):** determine as 5 UFs com maior quantidade de registros e suas participações percentuais.
2. **Visualização (Power BI ou Plotly):** crie um gráfico por UF, com cores por `DEPENDENCIA_CODIGO`, e explique como verificar a soma dos valores.
3. **Verificação:** compare o total exibido no gráfico com `df_powerbi['REGISTROS'].sum()`. Explique divergências, se ocorrerem.
4. **Interpretação:** por que contagem de registros não significa automaticamente quantidade de escolas únicas? Consulte o dicionário da base.
5. **Documentação:** descreva se a conexão com `QuickVisualize` funcionou, que erro ocorreu em caso negativo e qual alternativa foi utilizada.

**Entrega:** link do Colab com código (sem credenciais), gráficos, análise de resultados e justificativa das decisões de tratamento.

**Critério de sucesso:** o processamento em Python e a análise são obrigatórios; a criação nativa do relatório Power BI está condicionada à conta e à compatibilidade do ambiente. Não penalize estudantes por um bloqueio de licença ou política institucional.
